# CVAE-ViT — Full Retraining on 100% of the Dataset

**Purpose**: Retrain the conditional VAE with a Vision Transformer (ViT-B/16) encoder from scratch on the full magnetic-domain dataset (169,671 images).
**Inputs**: `dataset_unificado_v2.npz` (keys `img`, `params`, `labels`), shared `metrics.py` (Kaggle dataset `carloscanamejoy/physicalmetrics`).
**Outputs**: `OUTPUT_DIR/checkpoints/cvae_vit_best.h5` (best checkpoint, weights), `cvae_vit_best_weights.h5`, `cvae_vit_param_scaler.pkl`, training-history JSON, and publication figures in `OUTPUT_DIR/cvae_vit_training/`.
**Execution environment**: Kaggle / Google Colab (run only one setup cell).
**Dependencies**: tensorflow, transformers, scikit-learn, scikit-image, matplotlib, seaborn, scipy, tqdm, numpy

Architecture summary (estructura común a la variante CVAE-Xception):
- Encoder backbone: ViT-B/16 (12 transformer blocks), first 8 blocks frozen, blocks 9–12 trainable.
- Posterior network: GAP over the ViT token sequence (768) + condition embedding → Dense(512) → Dense(256) → [μ_q, log σ²_q] (z_dim = 128).
- Prior network p(z|θ): condition embedding (64) → Dense(64) → Dense(128) → [μ_p, log σ²_p] (128 units each).
- Decoder: z (128) ⊕ parameter embedding (64) → transposed convolutions 5×5 → 10×10 → 20×20 → 40×40 → 2 residual blocks → tanh output. **Emite 40×40** (sin crop interno).
- **Preprocesado del target (igual que `ddpm_spines_train`)**: la imagen real 39×39 se **pad-refleja a 40×40** (`tf.pad mode=REFLECT`, sin interpolación) para casar con la salida del decoder. El crop a 39×39 (`[:39,:39]`) se aplica **solo en evaluación/visualización**, preservando la textura pixel-a-pixel.
- Loss: `L = (1 − SSIM) + 0.0147·L1 + β·KLD(q ‖ p)`, calculada a 40×40. (El término de patch-variance se eliminó para unificar la estructura entre variantes.)
- Selección del best: **`val_loss`**.
- β-KLD schedule **F_plateau**: linear warmup from 1e-6 to β_max = 0.0657 over 5 epochs, then constant plateau (30 epochs total).

In [1]:
# ============================================================
# CELDA 0 — Instalar dependencias (ejecutar y REINICIAR RUNTIME)
# Menú → Runtime → Restart session, luego continuar desde celda 1
# ============================================================
!pip install -q "transformers==4.44.2" tf-keras
!pip install -q tensorflow scikit-learn scikit-image tqdm matplotlib seaborn scipy kaggle

import os
# Forzar que TF use Keras 2 (requerido por TFViTModel)
os.environ["TF_USE_LEGACY_KERAS"] = "1"
print("✅ Paquetes instalados. REINICIA EL RUNTIME antes de continuar.")

✅ Paquetes instalados. REINICIA EL RUNTIME antes de continuar.


## Google Colab Environment

In [2]:
# ============================================================
# CELDA 1 — ENVIRONMENT SETUP (GOOGLE COLAB)
# ============================================================
import os, sys, shutil
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # <-- ANTES de importar TF

from google.colab import files

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(os.path.join(kaggle_dir, "kaggle.json")):
    print("⬆️  Sube tu archivo kaggle.json:")
    uploaded = files.upload()
    shutil.move("kaggle.json", os.path.join(kaggle_dir, "kaggle.json"))
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
else:
    print("✅ kaggle.json ya existe")

# --- Download datasets ---
!kaggle datasets download -d carloscanamejoy/dataset-spines-united-v2   -p /content/data    --unzip
!kaggle datasets download -d carloscanamejoy/weights-xception-model      -p /content/weights --unzip
!kaggle datasets download -d carloscanamejoy/weights-models              -p /content/weights --unzip
!kaggle datasets download -d carloscanamejoy/weights-cvae-models         -p /content/weights --unzip
!kaggle datasets download -d carloscanamejoy/physicalmetrics             -p /content/weights --unzip

for tag, path in [
    ("Dataset",  "/content/data/dataset_unificado_v2.npz"),
    ("Metrics",  "/content/weights/metrics.py"),
]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"{status}  {tag}: {path}")

✅ kaggle.json ya existe
Dataset URL: https://www.kaggle.com/datasets/carloscanamejoy/dataset-spines-united-v2
License(s): unknown
100% 421M/421M [00:11<00:00, 37.7MB/s]

Dataset URL: https://www.kaggle.com/datasets/carloscanamejoy/weights-xception-model
License(s): apache-2.0
100% 670M/670M [00:17<00:00, 39.6MB/s]

Dataset URL: https://www.kaggle.com/datasets/carloscanamejoy/weights-models
License(s): apache-2.0
Resuming from 32505856 bytes (814049524 bytes left)...
100% 807M/807M [00:20<00:00, 39.8MB/s]

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
Dataset URL: https://www.kaggle.com/datasets/carloscanamejoy/physicalmetrics
License(s): unknown
100% 3.38k/3.38k [00:00<00:00, 12.1MB/s]

✅  Dataset: /content/data/dataset_unificado_v2.npz
✅  Metrics: /content/weights/metrics.py


## Load Shared Metrics

In [3]:
# ── Load shared metrics module ───────────────────────────────
import importlib.util, sys, os

_METRICS_COLAB  = "/content/weights/metrics.py"
_METRICS_LOCAL  = os.path.join("..", "utils", "metrics.py")
_metrics_path   = next(p for p in (_METRICS_COLAB, _METRICS_LOCAL)
                       if os.path.exists(p))

spec    = importlib.util.spec_from_file_location("metrics", _metrics_path)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules["metrics"] = metrics

from metrics import (
    STRUCTURE_MAP, STRUCTURE_NAMES, STRUCTURE_COLORS, MODEL_COLORS, PARAM_NAMES,
    circular_mask, MASK,
    masked_mse, masked_bce, masked_ssim,
    cosine_similarity_pair, cosine_similarity_batch,
    magnetization, abs_magnetization, cnn_correlation,
    structure_factor, azimuthal_average, oz_fit, chi_ensemble,
    shift_image, reflect_image,
    normalize_metrics,
    save_figure, apply_figure_style,
    center_crop, get_structure_label,
)
print(f"✅ metrics loaded from: {_metrics_path}")

✅ metrics loaded from: /content/weights/metrics.py


## Shared Configuration

Auto-detects the execution environment (`_ON_KAGGLE`) and defines every path used
by this notebook — the rest of the code never hardcodes paths.

In [ ]:
import json
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Paths (Colab) ---
DATASET_PATH  = "/content/data/dataset_unificado_v2.npz"
XCEPTION_PATH = "/content/weights/modelo_xception_fulldatabaseV3100.h5"
DDPM_PATH     = "/content/weights/ddpm_spines_final_39crop.pt"
CVAEXPN_PATH  = "/content/weights/cvae_xception.h5"
CVAEVIT_PATH  = "/content/weights/cvae_vit.h5"
METRICS_PATH  = "/content/weights/metrics.py"
OUTPUT_DIR    = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CKPT_DIR  = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
BEST_CKPT = os.path.join(CKPT_DIR, "cvae_vit_best.h5")

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

FIG_DIR = os.path.join(OUTPUT_DIR, "cvae_vit_training")
os.makedirs(FIG_DIR, exist_ok=True)

# --- Training hyperparameters ---
Z_DIM            = 128
COND_EMB_DIM     = 64
EPOCHS           = 30
BATCH_SIZE       = 32
LR               = 1e-4
BETA_START       = 1e-6
BETA_MAX         = 0.0657
WARMUP_EPOCHS    = 5
L1_WEIGHT        = 0.0147
KL_ACTIVE_THRESHOLD = 0.01

apply_figure_style()
print(f"TensorFlow {tf.__version__} | GPUs: {tf.config.list_physical_devices('GPU')}")

## Section 1 — Data Loading & Stratified Split

Same protocol as the CVAE-Xception notebook: 70/15/15 split, `SEED=42`, stratified
by magnetic structure label; θ MinMax-normalized with a train-fitted scaler.

In [5]:
data = np.load(DATASET_PATH)
print(f"npz keys: {data.files}")

imgs     = data["img"].astype(np.float32)
params   = data["params"].astype(np.float32)
clusters = data["labels"].astype(int)                # numeric cluster ids (see STRUCTURE_MAP)
if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]
N = len(imgs)
print(f"images {imgs.shape}  range [{imgs.min():.2f}, {imgs.max():.2f}] | "
      f"clusters {sorted(np.unique(clusters).tolist())}")

# 70/15/15 split, SEED=42, stratified by cluster label
all_idx = np.arange(N)
idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=SEED,
                                       stratify=clusters)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED,
                                     stratify=clusters[idx_temp])
print(f"train={len(idx_train):,}  val={len(idx_val):,}  test={len(idx_test):,}")

param_scaler = MinMaxScaler().fit(params[idx_train])
with open(os.path.join(OUTPUT_DIR, "cvae_vit_param_scaler.pkl"), "wb") as f:
    pickle.dump(param_scaler, f)

def make_tf_dataset(idx, shuffle):
    x = imgs[idx]
    y = param_scaler.transform(params[idx]).astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

ds_train = make_tf_dataset(idx_train, shuffle=True)
ds_val   = make_tf_dataset(idx_val, shuffle=False)

npz keys: ['img', 'params', 'labels', 'label_keys', 'label_names', 'column_names']
images (169671, 39, 39, 1)  range [-1.00, 1.00] | clusters [4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17]
train=118,769  val=25,451  test=25,451


## Section 2 — Model Definition

The ViT-B/16 backbone comes from HuggingFace `transformers` (`google/vit-base-patch16-224-in21k`).
Its checkpoint normalizes pixels to [-1, 1], which matches the dataset range directly.
The first 8 transformer blocks (and the patch/position embeddings) are frozen;
blocks 9–12 remain trainable. Features are the GAP over the output token sequence.

In [ ]:
!pip install -q "transformers==4.41.2"

from tensorflow.keras import layers
from transformers import TFViTModel

VIT_FEAT_DIM = 768
N_FROZEN_BLOCKS = 8

# Tamaño nativo del dataset y tamaño interno del decoder.
# Igual que en ddpm_spines_train: se entrena a 40×40 con la imagen real
# pad-reflejada 39→40 (sin interpolación), y la salida se cropea a 39×39
# solo en evaluación/visualización. Así se preserva la textura pixel-a-pixel.
NATIVE_SIZE = 39
MODEL_SIZE  = 40


def pad_reflect_39_to_40(x):
    """(B, 39, 39, 1) → (B, 40, 40, 1) con reflect en bordes inferior/derecho."""
    return tf.pad(x, [[0, 0], [0, 1], [0, 1], [0, 0]], mode="REFLECT")


class ViTBackbone(tf.keras.layers.Layer):
    """ViT-B/16 feature extractor: resize 39→224, RGB, GAP over tokens."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.vit.vit.embeddings.trainable = False
        for i, block in enumerate(self.vit.vit.encoder.layer):
            block.trainable = i >= N_FROZEN_BLOCKS
        n_tr = sum(int(b.trainable) for b in self.vit.vit.encoder.layer)
        print(f"ViT backbone: {n_tr}/{len(self.vit.vit.encoder.layer)} trainable transformer blocks")

    def call(self, x, training=False):
        x = tf.image.resize(x, (224, 224))                 # (B, 224, 224, 1)
        x = tf.image.grayscale_to_rgb(x)                   # (B, 224, 224, 3) in [-1, 1]
        x = tf.transpose(x, [0, 3, 1, 2])                  # ViT expects channels-first
        out = self.vit(pixel_values=x, training=training).last_hidden_state  # (B, 197, 768)
        return tf.reduce_mean(out, axis=1)                 # GAP over token sequence


def build_posterior():
    feat = layers.Input(shape=(VIT_FEAT_DIM,), name="features")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(COND_EMB_DIM, activation="relu", name="post_cond_emb")(cond)
    h = layers.Concatenate()([feat, c])
    h = layers.Dense(512, activation="relu")(h)
    h = layers.Dense(256, activation="relu")(h)
    mu      = layers.Dense(Z_DIM, name="mu_q")(h)
    log_var = layers.Dense(Z_DIM, name="log_var_q")(h)
    return tf.keras.Model([feat, cond], [mu, log_var], name="posterior_q")


def build_prior():
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(COND_EMB_DIM, activation="relu", name="prior_cond_emb")(cond)
    h = layers.Dense(64, activation="relu")(c)
    h = layers.Dense(128, activation="relu")(h)
    mu      = layers.Dense(Z_DIM, name="mu_p")(h)
    log_var = layers.Dense(Z_DIM, name="log_var_p")(h)
    return tf.keras.Model(cond, [mu, log_var], name="prior_p")


def _res_block(x, ch):
    h = layers.Conv2D(ch, 3, padding="same", activation="relu")(x)
    h = layers.Conv2D(ch, 3, padding="same")(h)
    return layers.Activation("relu")(layers.Add()([x, h]))


def build_decoder():
    """Decoder que emite 40×40 (sin crop interno). El crop a 39 es solo de evaluación."""
    z    = layers.Input(shape=(Z_DIM,), name="z")
    cond = layers.Input(shape=(8,), name="theta_norm")
    c = layers.Dense(COND_EMB_DIM, activation="relu", name="dec_cond_emb")(cond)
    h = layers.Concatenate()([z, c])                       # 128 + 64 = 192
    h = layers.Dense(5 * 5 * 256, activation="relu")(h)
    h = layers.Reshape((5, 5, 256))(h)
    h = layers.Conv2DTranspose(128, 4, strides=2, padding="same", activation="relu")(h)  # 10×10
    h = layers.Conv2DTranspose(64, 4, strides=2, padding="same", activation="relu")(h)   # 20×20
    h = layers.Conv2DTranspose(32, 4, strides=2, padding="same", activation="relu")(h)   # 40×40
    h = _res_block(h, 32)
    h = _res_block(h, 32)
    out = layers.Conv2D(1, 3, padding="same", activation="tanh")(h)                      # 40×40
    return tf.keras.Model([z, cond], out, name="decoder")

In [ ]:
class CVAEViT(tf.keras.Model):
    """Conditional VAE con encoder ViT y prior condicional aprendido.

    Estructura común a la variante Xception: loss = (1−SSIM) + λ_L1·L1 + β·KLD(q‖p),
    calculada a 40×40 contra la imagen real pad-reflejada (39→40). Sin término de
    patch-variance (se quitó para unificar la estructura entre variantes)."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.backbone  = ViTBackbone(name="vit_backbone")
        self.posterior = build_posterior()
        self.prior     = build_prior()
        self.decoder   = build_decoder()
        self.beta = tf.Variable(BETA_START, trainable=False, dtype=tf.float32, name="beta")
        names = ["loss", "ssim", "l1", "mse",
                 "kld_raw", "beta_kld", "var_q", "kl_active"]
        self.trackers = {n: tf.keras.metrics.Mean(name=n) for n in names}

    @property
    def metrics(self):
        return list(self.trackers.values())

    def call(self, inputs, training=False):
        x, y = inputs
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        eps = tf.random.normal(tf.shape(mu_q))
        z = mu_q + tf.exp(0.5 * log_var_q) * eps
        return self.decoder([z, y], training=training)     # (B, 40, 40, 1)

    def compute_losses(self, x, y, training):
        # x es 39×39 (entrada del backbone); el target se pad-refleja a 40×40
        # para casar con la salida del decoder, igual que en el DDPM.
        x40  = pad_reflect_39_to_40(x)
        feat = self.backbone(x, training=training)
        mu_q, log_var_q = self.posterior([feat, y], training=training)
        mu_p, log_var_p = self.prior(y, training=training)

        eps = tf.random.normal(tf.shape(mu_q))
        z = mu_q + tf.exp(0.5 * log_var_q) * eps
        x_hat = self.decoder([z, y], training=training)    # (B, 40, 40, 1)

        x01, xh01 = (x40 + 1.0) / 2.0, (x_hat + 1.0) / 2.0
        ssim_val = tf.reduce_mean(tf.image.ssim(x01, xh01, max_val=1.0))
        l1  = tf.reduce_mean(tf.abs(x40 - x_hat))
        mse = tf.reduce_mean(tf.square(x40 - x_hat))

        kld_units = 0.5 * (log_var_p - log_var_q
                           + (tf.exp(log_var_q) + tf.square(mu_q - mu_p)) / tf.exp(log_var_p)
                           - 1.0)
        kld_per_unit = tf.reduce_mean(kld_units, axis=0)
        kld = tf.reduce_sum(kld_per_unit)
        kl_active = tf.reduce_mean(tf.cast(kld_per_unit > KL_ACTIVE_THRESHOLD, tf.float32))

        loss = (1.0 - ssim_val) + L1_WEIGHT * l1 + self.beta * kld
        return {
            "loss": loss, "ssim": ssim_val, "l1": l1, "mse": mse,
            "kld_raw": kld, "beta_kld": self.beta * kld,
            "var_q": tf.reduce_mean(tf.exp(log_var_q)), "kl_active": kl_active,
        }

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            out = self.compute_losses(x, y, training=True)
        grads = tape.gradient(out["loss"], self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        for k, m in self.trackers.items():
            m.update_state(out[k])
        logs = {k: m.result() for k, m in self.trackers.items()}
        logs["kl_weight"] = self.beta
        return logs

    def test_step(self, data):
        x, y = data
        out = self.compute_losses(x, y, training=False)
        for k, m in self.trackers.items():
            m.update_state(out[k])
        return {k: m.result() for k, m in self.trackers.items()}

    def sample_from_prior(self, y_norm, n_samples=1):
        """Muestrea desde el prior p(z|θ). Devuelve 40×40 (cropear a 39 en eval)."""
        y = tf.repeat(tf.convert_to_tensor(y_norm, tf.float32), n_samples, axis=0)
        mu_p, log_var_p = self.prior(y, training=False)
        z = mu_p + tf.exp(0.5 * log_var_p) * tf.random.normal(tf.shape(mu_p))
        x = self.decoder([z, y], training=False)
        return tf.reshape(x, (len(y_norm), n_samples, MODEL_SIZE, MODEL_SIZE)).numpy()


cvae = CVAEViT(name="cvae_vit")
_ = cvae((imgs[idx_val[:2]], param_scaler.transform(params[idx_val[:2]]).astype(np.float32)))
cvae.compile(optimizer=tf.keras.optimizers.Adam(LR))
print(f"Total parameters: {cvae.count_params():,}")

## Section 3 — Callbacks: β schedule (F_plateau), checkpoints, snapshots

In [ ]:
def beta_for_epoch(epoch, total_epochs=EPOCHS, beta_start=BETA_START,
                   beta_max=BETA_MAX, warmup_epochs=WARMUP_EPOCHS):
    """
    F_plateau schedule.
    Warmup: linearly ramp from beta_start to beta_max over warmup_epochs.
    Plateau: hold at beta_max for remaining epochs.
    """
    if epoch <= warmup_epochs:
        p = (epoch - 1) / max(1, warmup_epochs - 1)
        return beta_start + (beta_max - beta_start) * p
    else:
        return beta_max


class BetaScheduler(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.model.beta.assign(beta_for_epoch(epoch + 1))


SNAPSHOT_EPOCHS = {1, 10, 20, 30}
rng = np.random.RandomState(SEED)
snap_idx = rng.choice(idx_val, 5, replace=False)
snap_x = imgs[snap_idx]                                       # 39×39 (original)
snap_y = param_scaler.transform(params[snap_idx]).astype(np.float32)


class ReconSnapshot(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        e = epoch + 1
        if e not in SNAPSHOT_EPOCHS:
            return
        # La salida del decoder es 40×40 → cropear a 39×39 para comparar con el original.
        x_hat = self.model((snap_x, snap_y), training=False).numpy()[:, :NATIVE_SIZE, :NATIVE_SIZE, :]
        fig, axes = plt.subplots(5, 3, figsize=(7, 11))
        for i in range(5):
            for j, (img, title) in enumerate([
                    (snap_x[i, :, :, 0], "Original"),
                    (x_hat[i, :, :, 0], "Reconstruction"),
                    (np.abs(snap_x[i, :, :, 0] - x_hat[i, :, :, 0]), "Difference")]):
                ax = axes[i, j]
                cmap, vmin, vmax = ("jet", -1, 1) if j < 2 else ("hot", 0, 1)
                ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
                ax.set_xticks([]); ax.set_yticks([])
                if i == 0:
                    ax.set_title(title)
        fig.suptitle(f"CVAE-ViT reconstructions — epoch {e}")
        save_figure(fig, os.path.join(FIG_DIR, f"reconstructions_epoch_{e:02d}"))
        plt.close(fig)


ckpt_best = tf.keras.callbacks.ModelCheckpoint(
    BEST_CKPT,
    monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", patience=5, factor=0.5, verbose=1)

## Section 4 — Training (30 epochs, batch 32)

In [9]:
history = cvae.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    callbacks=[BetaScheduler(), ReconSnapshot(), ckpt_best, reduce_lr],
    verbose=1,
)

cvae.load_weights(BEST_CKPT)
cvae.save_weights(os.path.join(CKPT_DIR, "cvae_vit_best_weights.h5"))
hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(os.path.join(OUTPUT_DIR, "cvae_vit_history.json"), "w") as f:
    json.dump(hist, f, indent=2)
print(f"Saved: {BEST_CKPT}, cvae_vit_best_weights.h5, cvae_vit_history.json")

Epoch 1/30


3712/3712 [==============================] - ETA: 0s - loss: 0.6497 - ssim: 0.3950 - l1: 0.2866 - mse: 0.1887 - patch_var_mse: 0.0044 - kld_raw: 354.7903 - beta_kld: 3.5479e-04 - var_q: 0.2883 - kl_active: 0.9942 - kl_weight: 1.0000e-06
Epoch 1: val_loss improved from inf to 0.50996, saving model to /content/outputs/checkpoints/cvae_vit_best.h5
3712/3712 [==============================] - 400s 100ms/step - loss: 0.6497 - ssim: 0.3950 - l1: 0.2866 - mse: 0.1887 - patch_var_mse: 0.0044 - kld_raw: 354.7903 - beta_kld: 3.5479e-04 - var_q: 0.2883 - kl_active: 0.9942 - kl_weight: 1.0000e-06 - val_loss: 0.5100 - val_ssim: 0.5239 - val_l1: 0.2419 - val_mse: 0.1417 - val_patch_var_mse: 0.0033 - val_kld_raw: 407.5032 - val_beta_kld: 4.0750e-04 - val_var_q: 0.0240 - val_kl_active: 1.0000 - lr: 1.0000e-04
Epoch 2/30
3712/3712 [==============================] - ETA: 0s - loss: 0.7027 - ssim: 0.4175 - l1: 0.3407 - mse: 0.2242 - patch_var_mse: 0.0048 - kld_raw: 4.3939 - beta_kld: 0.0722 - var_q: 0.85

In [ ]:
# ── Continuar entrenamiento desde el último checkpoint ───────
RESUME_CKPT    = BEST_CKPT                        # o la ruta que tengas
EXTRA_EPOCHS   = 21                                # ajustá a gusto
RESUME_LR      = 1e-5                              # LR bajo para no romper lo aprendido

# 1. Cargar pesos guardados
cvae.load_weights(RESUME_CKPT)
print(f"✅ Pesos cargados desde: {RESUME_CKPT}")

# 2. Recompilar con LR fresco (el ReduceLR puede haberlo bajado demasiado)
cvae.compile(optimizer=tf.keras.optimizers.Adam(RESUME_LR))

# 3. Fijar beta en el plateau — el warmup ya pasó
cvae.beta.assign(BETA_MAX)
print(f"β fijado en {BETA_MAX}")

# 4. Callbacks SIN save_best_only — guardar cada mejora Y el último
RESUME_CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints_resume")
os.makedirs(RESUME_CKPT_DIR, exist_ok=True)

ckpt_all = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(RESUME_CKPT_DIR, "cvae_vit_epoch_{epoch:03d}.h5"),
    save_weights_only=True, verbose=1)                     # guarda TODAS las épocas

ckpt_best_resume = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(RESUME_CKPT_DIR, "cvae_vit_best_resume.h5"),
    monitor="val_loss", save_best_only=True,
    save_weights_only=True, verbose=1)

reduce_lr_resume = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", patience=8, factor=0.5,
    min_lr=1e-7, verbose=1)

SNAPSHOT_EPOCHS_RESUME = set(range(10, EXTRA_EPOCHS + 1, 10))  # cada 10

class ReconSnapshotResume(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        e = epoch + 1
        if e not in SNAPSHOT_EPOCHS_RESUME:
            return
        # Salida 40×40 → cropear a 39×39 para comparar con el original.
        x_hat = self.model((snap_x, snap_y), training=False).numpy()[:, :NATIVE_SIZE, :NATIVE_SIZE, :]
        fig, axes = plt.subplots(5, 3, figsize=(7, 11))
        for i in range(5):
            for j, (img, title) in enumerate([
                    (snap_x[i, :, :, 0], "Original"),
                    (x_hat[i, :, :, 0], "Reconstruction"),
                    (np.abs(snap_x[i, :, :, 0] - x_hat[i, :, :, 0]), "Difference")]):
                ax = axes[i, j]
                cmap, vmin, vmax = ("jet", -1, 1) if j < 2 else ("hot", 0, 1)
                ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
                ax.set_xticks([]); ax.set_yticks([])
                if i == 0:
                    ax.set_title(title)
        fig.suptitle(f"CVAE-ViT resume — epoch {e}")
        save_figure(fig, os.path.join(FIG_DIR, f"resume_recon_epoch_{e:02d}"))
        plt.close(fig)

# 5. Entrenar
history_resume = cvae.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EXTRA_EPOCHS,
    callbacks=[ReconSnapshotResume(), ckpt_all, ckpt_best_resume, reduce_lr_resume],
    verbose=1,
)

# 6. Guardar el último (no necesariamente el "best") + historial
cvae.save_weights(os.path.join(RESUME_CKPT_DIR, "cvae_vit_last.h5"))
hist_resume = {k: [float(v) for v in vals] for k, vals in history_resume.history.items()}
with open(os.path.join(OUTPUT_DIR, "cvae_vit_history_resume.json"), "w") as f:
    json.dump(hist_resume, f, indent=2)
print("✅ Guardado: last weights + best_resume + history_resume.json")

## Section 5 — Training Curves & β Schedule

In [16]:
# Unir historiales para graficar completo
hist_full = {}
for k in hist:
    hist_full[k] = hist[k] + hist_resume.get(k, [])
epochs_ax = np.arange(1, len(hist_full["loss"]) + 1)

In [ ]:
epochs_ax = np.arange(1, len(hist_full["loss"]) + 1)

# Curvas estándar (idénticas en CVAE-ViT y CVAE-Xception para comparación directa).
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
panels = [
    ("Total composite loss", [("loss", "train"), ("val_loss", "val")]),
    ("SSIM (reconstruction quality)", [("ssim", "train"), ("val_ssim", "val")]),
    ("L1 term", [("l1", "train"), ("val_l1", "val")]),
    ("Raw KLD", [("kld_raw", "train"), ("val_kld_raw", "val")]),
    ("β·KLD", [("beta_kld", "train"), ("val_beta_kld", "val")]),
    ("β schedule (F_plateau)", [("beta", None)]),
]
for ax, (title, series) in zip(axes.flat, panels):
    for key, lbl in series:
        if key == "beta":
            ax.plot(epochs_ax, [beta_for_epoch(e) for e in epochs_ax], color="#4C72B0")
        elif key in hist_full:
            ax.plot(epochs_ax, hist_full[key], label=lbl)
    ax.set_title(title); ax.set_xlabel("Epoch")
    if any(lbl for _, lbl in series):
        ax.legend()
    ax.grid(alpha=.3)
fig.suptitle("CVAE-ViT — training curves")
plt.tight_layout()
save_figure(fig, os.path.join(FIG_DIR, "training_curves"))
plt.show()

In [ ]:
# Diagnósticos del espacio latente (idénticos en ambas variantes).
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_ax, hist_full["kl_active"], label="train")
if "val_kl_active" in hist_full:
    axes[0].plot(epochs_ax, hist_full["val_kl_active"], label="val")
axes[0].axhline(.5, color="gray", ls="--")
axes[0].set_title("Fraction of active latent units (KLD > 0.01)")
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(epochs_ax, hist_full["var_q"], label="train")
if "val_var_q" in hist_full:
    axes[1].plot(epochs_ax, hist_full["val_var_q"], label="val")
axes[1].axhline(1.0, color="gray", ls="--")
axes[1].set_title("Mean posterior variance var_q")
axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=.3)
fig.suptitle("CVAE-ViT — latent-space diagnostics")
plt.tight_layout()
save_figure(fig, os.path.join(FIG_DIR, "latent_diagnostics"))
plt.show()

## Section 6 — Post-training Sanity Check: Prior Sampling

In [ ]:
# Panel de 3 filas: Original (39×39) / Reconstruida (posterior) / Generada (prior).
# Ambas salidas son 40×40 → se cropean a 39×39 para comparar con el original.
cvae.load_weights(BEST_CKPT)
n_show = 8
rng_show = np.random.RandomState(SEED)
show_idx = rng_show.choice(idx_test, n_show, replace=False)

x_orig = imgs[show_idx]                                       # (n, 39, 39, 1)
y_in   = param_scaler.transform(params[show_idx]).astype(np.float32)

x_recon = cvae((x_orig, y_in), training=False).numpy()[:, :NATIVE_SIZE, :NATIVE_SIZE, :]   # posterior
x_gen   = cvae.sample_from_prior(y_in, n_samples=1)[:, 0, :NATIVE_SIZE, :NATIVE_SIZE]       # prior

fig, axes = plt.subplots(3, n_show, figsize=(2.4 * n_show, 7.4))
row_imgs   = [x_orig[:, :, :, 0], x_recon[:, :, :, 0], x_gen]
row_labels = ["Original", "Reconstruida", "Generada"]
for r, (imgs_row, lbl) in enumerate(zip(row_imgs, row_labels)):
    for c in range(n_show):
        ax = axes[r, c]
        ax.imshow(imgs_row[c], cmap="jet", vmin=-1, vmax=1, interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(lbl, fontsize=11)
        if r == 0:
            ax.set_title(f"T={params[show_idx[c]][0]:.1f}", fontsize=8)
fig.suptitle("CVAE-ViT — Original / Reconstrucción (posterior) / Generación (prior)")
plt.tight_layout()
save_figure(fig, os.path.join(FIG_DIR, "orig_recon_gen"))
plt.show()

## After Training — Upload Weights to Kaggle

After training completes, the best checkpoint is saved at:
- Kaggle: `/kaggle/working/outputs/checkpoints/cvae_vit_best.h5`
- Colab:  `/content/outputs/checkpoints/cvae_vit_best.h5`

Upload this file to the Kaggle dataset `carloscanamejoy/weights-cvae-models` before
running `generative_comparison.ipynb` or `ciclo_completo_v2.ipynb`.

> **Note**: the evaluation notebooks expect the file inside that dataset to be named
> `cvae_vit.h5` (see `CVAEVIT_PATH`) — rename `cvae_vit_best.h5` to `cvae_vit.h5`
> when uploading.